# BankAssist Triage — simple CI plan

**CI (continuous integration)** means GitHub checks a code change automatically. This notebook documents the workflow now stored in `.github/workflows/ci.yml`. It contains no executable cells.

For this project, the smallest useful check is: install the project's pinned dependencies, run the existing tests, and verify that the Docker image builds. A passing run is evidence of automation and basic reproducibility. It does not prove the live app or model is accurate.

## 1. Implemented workflow

The repository now contains **`.github/workflows/ci.yml`** with the YAML below. Once this file is committed and pushed, GitHub Actions will run it on pushes and pull requests. Merely viewing or running this notebook does not trigger CI.

```yaml
name: CI

on: [push, pull_request]

permissions:
  contents: read

jobs:
  test-and-build:
    runs-on: ubuntu-latest
    steps:
      - name: Get project files
        uses: actions/checkout@v7

      - name: Set up Python
        uses: actions/setup-python@v7
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: python -m pip install -r requirements.txt -r requirements-dev.txt

      - name: Run tests with the packaged model
        env:
          MODEL_SOURCE: local
        run: python -m pytest -q

      - name: Check Docker build
        run: docker build -f docker/Dockerfile -t bankassist-ci .
```

## 2. What each step checks

1. **Get project files:** GitHub checks out the committed repository.
2. **Set up Python and install dependencies:** CI uses a fresh Python 3.11 environment, matching the Dockerfile, and the project's requirements files.
3. **Run tests:** `MODEL_SOURCE=local` makes the API load the packaged `models/champion_model.pkl` without needing a personal MLflow registry on the GitHub runner. The existing tests check the API behaviour.
4. **Check Docker build:** verifies that the Dockerfile can package the committed project. This builds an image; it does not start or deploy a service.

The workflow does **not** retrain the model, change `@champion`, publish an image, or deploy to Render. The existing prediction test was corrected to check the API's `route` response field instead of the obsolete `destination` field. `httpx` was added to `requirements-dev.txt` because FastAPI's test client needs it.

## 3. How to use this for the presentation

1. Commit and push `.github/workflows/ci.yml` and the corrected prediction test to GitHub.
2. Open the repository's **Actions** tab and inspect the `CI` run. If a step fails, read its log and fix the specific cause before claiming CI works.
3. Show a successful run or screenshot and say: *Each push automatically runs our tests and checks that the Docker image builds.*

**Local verification:** with `MODEL_SOURCE=local` and the project's pinned scikit-learn version, both existing tests passed (2 passed). The Docker image also built successfully. These checks were run locally; no GitHub Actions run has been observed yet.

**Status:** workflow file created locally. CI is only demonstrated once it has been pushed and a GitHub Actions run passes.